# 50. CUDA Graph Serving Dispatch：怎样在静态地址约束下安全复用捕获图？

## 面试回答主线

CUDA Graph 将一串 GPU kernel launch 预先捕获并重复 replay，以减少 CPU 调度开销，但它要求形状、控制流和内存地址稳定。LLM Serving 因请求长度、batch、adapter 与 KV 状态不同，不能用一张图覆盖所有请求；需要按明确的 dispatch key 分桶，并把动态输入复制进预分配静态 buffer。面试中我会先用动态 eager 基线，再实现固定 bucket、静态指针、padding mask 和图缓存命中轨迹。缓存键遗漏 LoRA adapter、dtype 或模型版本会复用错误权重，得到数值合法但语义错误的输出。真实 CUDA 环境还要在 warmup stream 中捕获，并为不支持的动态分支可靠回退 eager。这个 CPU 小实验不冒充 GPU 性能测试，只把安全调度合同变成可观察状态。

## 1. 真实案例：六个不同长度与 adapter 的在线请求

请求来自客服和金融两个 adapter，token ID 对应退款、账单、风控等真实短句。长度被映射到 4 或 8 的静态 bucket；相同 bucket 与 adapter 才允许复用捕获图。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示服务请求
import torch  # 导入 PyTorch 构造静态输入 buffer 与真实张量 forward
torch.set_num_threads(1)  # 限制 CPU 线程以保持教学执行稳定
requests = [{"id": "G01", "text": "订单 可以 退款", "tokens": [3, 5, 7], "adapter": "support"}, {"id": "G02", "text": "发票 抬头 修改 今天", "tokens": [2, 4, 6, 8], "adapter": "support"}, {"id": "G03", "text": "基金 风险 等级 如何 查询", "tokens": [9, 3, 4, 6, 2], "adapter": "finance"}, {"id": "G04", "text": "转账 失败 是否 触发 风控 审核 记录", "tokens": [8, 7, 5, 3, 9, 2, 4], "adapter": "finance"}, {"id": "G05", "text": "客服 查找 订单 并 创建 工单", "tokens": [6, 2, 5, 8, 3, 7], "adapter": "support"}, {"id": "G06", "text": "账单 怎么 导出", "tokens": [4, 9, 5], "adapter": "finance"}]  # 定义六个长度和 adapter 组合不同的真实请求
adapter_scale = {"support": 1.0, "finance": 1.5}  # 用不同缩放模拟两个 adapter 的权重差异
def choose_bucket(length):  # 将动态序列长度映射到可捕获的静态形状
    return 4 if length <= 4 else 8  # 本实验提供长度四和八两种图 bucket
preview = [{"请求": item["id"], "文本": item["text"], "长度": len(item["tokens"]), "bucket": choose_bucket(len(item["tokens"])), "adapter": item["adapter"]} for item in requests]  # 汇总图分派需要观察的请求属性
print("CUDA Graph Dispatch 输入预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示长度分桶与 adapter 组合

CUDA Graph Dispatch 输入预览：
[{'请求': 'G01', '文本': '订单 可以 退款', '长度': 3, 'bucket': 4, 'adapter': 'support'},
 {'请求': 'G02', '文本': '发票 抬头 修改 今天', '长度': 4, 'bucket': 4, 'adapter': 'support'},
 {'请求': 'G03',
  '文本': '基金 风险 等级 如何 查询',
  '长度': 5,
  'bucket': 8,
  'adapter': 'finance'},
 {'请求': 'G04',
  '文本': '转账 失败 是否 触发 风控 审核 记录',
  '长度': 7,
  'bucket': 8,
  'adapter': 'finance'},
 {'请求': 'G05',
  '文本': '客服 查找 订单 并 创建 工单',
  '长度': 6,
  'bucket': 8,
  'adapter': 'support'},
 {'请求': 'G06', '文本': '账单 怎么 导出', '长度': 3, 'bucket': 4, 'adapter': 'finance'}]


## 2. Baseline（基线）：每个动态请求都走 eager launch

这里用“有效 token 加权和”代表一个确定性的模型 forward，adapter 决定权重缩放。eager 每次都按实际长度创建张量，数值正确，但每个请求都支付固定 launch 调度成本。延迟数值是教学成本模型，不是伪造的 GPU benchmark。

In [2]:
def eager_forward(item):  # 对一个动态长度请求执行基线张量计算
    tokens = torch.tensor(item["tokens"], dtype=torch.float32)  # 每次请求创建实际长度输入张量
    score = float(tokens.sum() * adapter_scale[item["adapter"]])  # 用当前 adapter 权重计算确定性输出分数
    latency_units = 2.5 + 0.08 * len(item["tokens"])  # 构造包含固定 launch 开销的教学成本单位
    return score, latency_units  # 返回正确输出与基线调度成本
baseline_rows = []  # 收集六个动态请求的 eager 结果
for item in requests:  # 逐个执行相同的在线请求
    score, latency = eager_forward(item)  # 运行动态形状基线 forward
    baseline_rows.append({"请求": item["id"], "adapter": item["adapter"], "输出分数": score, "成本单位": round(latency, 2)})  # 保存逐请求基线指标
print("动态 eager 基线：")  # 标注当前输出属于基线
pprint(baseline_rows, sort_dicts=False)  # 展示每个请求的正确输出和固定 launch 成本

动态 eager 基线：
[{'请求': 'G01', 'adapter': 'support', '输出分数': 15.0, '成本单位': 2.74},
 {'请求': 'G02', 'adapter': 'support', '输出分数': 20.0, '成本单位': 2.82},
 {'请求': 'G03', 'adapter': 'finance', '输出分数': 36.0, '成本单位': 2.9},
 {'请求': 'G04', 'adapter': 'finance', '输出分数': 57.0, '成本单位': 3.06},
 {'请求': 'G05', 'adapter': 'support', '输出分数': 31.0, '成本单位': 2.98},
 {'请求': 'G06', 'adapter': 'finance', '输出分数': 27.0, '成本单位': 2.74}]


## 3. 手写静态图对象：预分配地址、copy_ 输入与 padding mask

真实 CUDA Graph 捕获后不能更换输入地址。下面的模拟图在构造时分配固定 bucket buffer，replay 时只用 `copy_` 更新内容；padding mask 保证补零位置不参与 forward。`data_ptr()` 让地址稳定性可以直接观察。

In [3]:
class CapturedGraphSimulator:  # 定义遵守静态形状和地址合同的图对象
    def __init__(self, bucket, adapter):  # 为一个固定 bucket 与 adapter 捕获执行状态
        self.bucket = bucket  # 保存捕获时的静态序列长度
        self.adapter = adapter  # 保存捕获时绑定的 adapter 权重版本
        self.input_buffer = torch.zeros(bucket, dtype=torch.float32)  # 预分配地址稳定的 token 输入 buffer
        self.mask_buffer = torch.zeros(bucket, dtype=torch.float32)  # 预分配地址稳定的 padding mask
        self.pointer = self.input_buffer.data_ptr()  # 记录捕获图依赖的原始输入地址
    def replay(self, tokens):  # 将一个兼容请求复制到静态 buffer 并执行图 replay
        padded = torch.zeros(self.bucket, dtype=torch.float32)  # 构造与捕获形状完全一致的临时输入
        mask = torch.zeros(self.bucket, dtype=torch.float32)  # 构造区分真实 token 与 padding 的临时 mask
        padded[:len(tokens)] = torch.tensor(tokens, dtype=torch.float32)  # 把动态 token 写入静态形状前缀
        mask[:len(tokens)] = 1.0  # 标记真实 token 位置以屏蔽 padding
        self.input_buffer.copy_(padded)  # 原地复制输入而不改变捕获时内存地址
        self.mask_buffer.copy_(mask)  # 原地复制 mask 保持图依赖地址稳定
        score = (self.input_buffer * self.mask_buffer).sum() * adapter_scale[self.adapter]  # 在静态 buffer 上执行确定性 forward
        return float(score), padded.tolist(), self.input_buffer.data_ptr()  # 返回输出、补齐输入和 replay 后地址
class GraphDispatcher:  # 定义按静态合同选择与缓存捕获图的分派器
    def __init__(self):  # 初始化空图缓存
        self.cache = {}  # 用完整 dispatch key 索引已捕获图
    def dispatch(self, item):  # 为一个动态请求选择安全图并执行 replay
        bucket = choose_bucket(len(item["tokens"]))  # 先把实际长度映射到静态 bucket
        key = (bucket, item["adapter"])  # 把形状与 adapter 一起写入缓存键
        cache_hit = key in self.cache  # 记录本次是否复用已经捕获的图
        if not cache_hit:  # 缓存缺失时需要创建匹配合同的新图
            self.cache[key] = CapturedGraphSimulator(bucket, item["adapter"])  # 捕获并缓存固定形状与权重的图对象
        graph = self.cache[key]  # 取得与请求合同完全一致的图
        score, padded, pointer_after = graph.replay(item["tokens"])  # 将动态数据复制进静态地址并 replay
        return {"key": key, "hit": cache_hit, "score": score, "padded": padded, "pointer_before": graph.pointer, "pointer_after": pointer_after}  # 返回可审计的分派轨迹
graph_demo = CapturedGraphSimulator(bucket=4, adapter="support")  # 创建一个长度四的客服图观察地址合同
pointer_before = graph_demo.input_buffer.data_ptr()  # 读取捕获前静态输入地址
demo_score, demo_padded, pointer_after = graph_demo.replay([3, 5, 7])  # 把三 token 请求复制到四长度 buffer
print({"静态地址_前": pointer_before, "静态地址_后": pointer_after, "padding后输入": demo_padded, "mask": graph_demo.mask_buffer.tolist(), "输出": demo_score})  # 展示 copy_ 后地址不变且 padding 被屏蔽

{'静态地址_前': 107542080, '静态地址_后': 107542080, 'padding后输入': [3.0, 5.0, 7.0, 0.0], 'mask': [1.0, 1.0, 1.0, 0.0], '输出': 15.0}


## 4. 完整 dispatch 轨迹：key、命中、地址与输出

分派器严格使用 `(bucket, adapter)` 作为最小 key。相同组合的后续请求命中缓存，不同 adapter 即使形状相同也会创建独立图；每次 replay 的静态地址保持不变。

In [4]:
dispatcher = GraphDispatcher()  # 创建空的安全图分派器
dispatch_rows = []  # 收集六个请求的图选择与 replay 轨迹
for item in requests:  # 按到达顺序处理真实在线请求
    trace = dispatcher.dispatch(item)  # 选择兼容图并执行静态 buffer replay
    dispatch_rows.append({"请求": item["id"], "dispatch_key": trace["key"], "缓存命中": trace["hit"], "padding输入": trace["padded"], "地址稳定": trace["pointer_before"] == trace["pointer_after"], "输出分数": trace["score"]})  # 保存逐请求可解释轨迹
print("安全 CUDA Graph 分派轨迹：")  # 输出核心调度结果标题
pprint(dispatch_rows, sort_dicts=False)  # 展示每个请求的图 key、命中与静态地址

安全 CUDA Graph 分派轨迹：
[{'请求': 'G01',
  'dispatch_key': (4, 'support'),
  '缓存命中': False,
  'padding输入': [3.0, 5.0, 7.0, 0.0],
  '地址稳定': True,
  '输出分数': 15.0},
 {'请求': 'G02',
  'dispatch_key': (4, 'support'),
  '缓存命中': True,
  'padding输入': [2.0, 4.0, 6.0, 8.0],
  '地址稳定': True,
  '输出分数': 20.0},
 {'请求': 'G03',
  'dispatch_key': (8, 'finance'),
  '缓存命中': False,
  'padding输入': [9.0, 3.0, 4.0, 6.0, 2.0, 0.0, 0.0, 0.0],
  '地址稳定': True,
  '输出分数': 36.0},
 {'请求': 'G04',
  'dispatch_key': (8, 'finance'),
  '缓存命中': True,
  'padding输入': [8.0, 7.0, 5.0, 3.0, 9.0, 2.0, 4.0, 0.0],
  '地址稳定': True,
  '输出分数': 57.0},
 {'请求': 'G05',
  'dispatch_key': (8, 'support'),
  '缓存命中': False,
  'padding输入': [6.0, 2.0, 5.0, 8.0, 3.0, 7.0, 0.0, 0.0],
  '地址稳定': True,
  '输出分数': 31.0},
 {'请求': 'G06',
  'dispatch_key': (4, 'finance'),
  '缓存命中': False,
  'padding输入': [4.0, 9.0, 5.0, 0.0],
  '地址稳定': True,
  '输出分数': 27.0}]


## 5. 结果解读：先验证数值一致，再讨论调度收益

图 replay 必须与 eager 输出逐请求完全一致，否则 launch 减少没有意义。成本模型对图首次捕获计 1.5、每次 replay 计 0.25；它仅说明缓存复用逻辑，不代表真实毫秒。表格同时输出数值一致性与命中状态。

In [5]:
comparison = []  # 构造 eager 与静态图方案的逐请求对照
for baseline, graph_row in zip(baseline_rows, dispatch_rows):  # 对齐同一请求的两种执行路径
    comparison.append({"请求": baseline["请求"], "eager输出": baseline["输出分数"], "graph输出": graph_row["输出分数"], "数值一致": baseline["输出分数"] == graph_row["输出分数"], "图缓存命中": graph_row["缓存命中"]})  # 保存质量与缓存指标
eager_cost = sum(row["成本单位"] for row in baseline_rows)  # 汇总动态 eager 的教学成本
capture_count = sum(not row["缓存命中"] for row in dispatch_rows)  # 统计唯一 dispatch key 触发的捕获次数
graph_cost = capture_count * 1.5 + len(requests) * 0.25  # 计算首次捕获与 replay 组成的图调度成本
print("同请求数值与调度对照：")  # 输出结果解读标题
pprint(comparison, sort_dicts=False)  # 展示每条请求先满足正确性再获得缓存命中
print(f"教学成本单位：eager={eager_cost:.2f}，graph={graph_cost:.2f}，捕获图数量={capture_count}")  # 汇总受控成本模型中的分派收益

同请求数值与调度对照：
[{'请求': 'G01', 'eager输出': 15.0, 'graph输出': 15.0, '数值一致': True, '图缓存命中': False},
 {'请求': 'G02', 'eager输出': 20.0, 'graph输出': 20.0, '数值一致': True, '图缓存命中': True},
 {'请求': 'G03', 'eager输出': 36.0, 'graph输出': 36.0, '数值一致': True, '图缓存命中': False},
 {'请求': 'G04', 'eager输出': 57.0, 'graph输出': 57.0, '数值一致': True, '图缓存命中': True},
 {'请求': 'G05', 'eager输出': 31.0, 'graph输出': 31.0, '数值一致': True, '图缓存命中': False},
 {'请求': 'G06', 'eager输出': 27.0, 'graph输出': 27.0, '数值一致': True, '图缓存命中': False}]
教学成本单位：eager=17.24，graph=7.50，捕获图数量=4


## 6. 失败案例与修正：缓存键遗漏 adapter 导致权重复用错误

错误分派器只按 bucket 缓存。客服图先占据长度四 key 后，金融请求 G06 会复用客服 scale=1.0，而不是金融 scale=1.5；输出仍是普通浮点数，很容易静默进入下游。修正是把 adapter、dtype、模型版本、decode/prefill 模式等所有会改变执行图的字段加入 key。

In [6]:
bad_cache = {}  # 初始化故意遗漏 adapter 维度的错误图缓存
def bad_dispatch(item):  # 定义只按形状复用捕获图的错误分派逻辑
    bucket = choose_bucket(len(item["tokens"]))  # 仍然正确计算静态序列 bucket
    if bucket not in bad_cache:  # 只检查长度而忽略 adapter 权重差异
        bad_cache[bucket] = CapturedGraphSimulator(bucket, item["adapter"])  # 首个请求决定该长度图绑定的 adapter
    graph = bad_cache[bucket]  # 取得可能绑定错误权重的捕获图
    score, padded, pointer = graph.replay(item["tokens"])  # 在错误 adapter 图上执行数值合法的 replay
    return score, graph.adapter  # 返回错误输出和实际绑定权重供诊断
support_score, support_binding = bad_dispatch(requests[0])  # 先让客服请求占据长度四图缓存
wrong_finance_score, reused_binding = bad_dispatch(requests[-1])  # 再让同 bucket 金融请求错误复用客服图
correct_finance_score, correct_latency = eager_forward(requests[-1])  # 用正确金融 adapter 计算权威输出
fixed_trace = GraphDispatcher().dispatch(requests[-1])  # 使用完整 key 的分派器重新执行金融请求
print({"失败请求": requests[-1]["id"], "错误复用adapter": reused_binding, "错误输出": wrong_finance_score, "正确输出": correct_finance_score, "完整key修正输出": fixed_trace["score"], "修正key": fixed_trace["key"]})  # 展示静默权重串用与修复结果

{'失败请求': 'G06', '错误复用adapter': 'support', '错误输出': 18.0, '正确输出': 27.0, '完整key修正输出': 27.0, '修正key': (4, 'finance')}


## 7. 生产差距与最小回归检查

真实 CUDA Graph 还要 warmup、处理随机数状态、固定 workspace、KV page table 与 batch 指针，并在显存压力下淘汰图缓存。prefill、decode、speculative decode 和不同量化 kernel 往往需要不同 key；动态不兼容请求必须走 eager 回退。性能结论必须在真实 GPU 上报告 TTFT、TPOT、P50/P99 与吞吐。最后的断言只验证本实验已经展示的地址、mask、数值等价和 adapter 隔离。

In [7]:
assert len(requests) >= 5  # 确认真实服务请求数量满足逐样本教学要求
assert pointer_before == pointer_after  # 确认 replay 通过 copy_ 保持静态输入地址不变
assert demo_padded[-1] == 0.0  # 确认短请求被补齐到捕获图的静态形状
assert all(row["地址稳定"] for row in dispatch_rows)  # 确认每次安全分派都遵守捕获地址合同
assert all(row["数值一致"] for row in comparison)  # 确认图 replay 与 eager 基线逐请求输出一致
assert wrong_finance_score != correct_finance_score  # 确认遗漏 adapter 的缓存键真实产生错误输出
assert fixed_trace["score"] == correct_finance_score  # 确认完整 dispatch key 恢复正确金融权重
print("回归检查通过：静态地址、padding mask、图缓存与 adapter 隔离均已验证。")  # 输出最终验收结论

回归检查通过：静态地址、padding mask、图缓存与 adapter 隔离均已验证。
